<a href="https://colab.research.google.com/github/HuchpechTW/public-colabs/blob/main/SPICEUP_HSVZ_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [27]:
from google.colab import auth
from google.auth import default
import gspread
import pandas as pd

# 1. 读取你云盘里的原始数据 (请根据实际文件名修改路径)
# 假设 Ads 表格里有 'Item ID', 'Cost', 'Conversion Value' 三列
performance_excel='Item-Performance-260126-260225'

# 从gmc 到处的含有item group id 的数据
item_group_excel='gmc_products_2026-02-28.tsv'

# 从shopify到处的product id销售数据
shop_excel='product_sales_20260126_20260225.csv'



# 1. 验证你的账号权限（运行会弹窗让你点授权）
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 2. 读取ads item performance数据
worksheet = gc.open(performance_excel).sheet1
rows = worksheet.get_all_values()
ads_df = pd.DataFrame.from_records(rows[1:], columns=rows[0])
ads_df['Item ID'] = ads_df['Item ID'].astype(str)
print('Google Ads item performancedata')
print(ads_df.head()) # 打印出来看看，瞬间搞定


# 3. 读取gmc item group id数据
# 假设 GMC 表格里有 'id', 'item_group_id' 两列
gmc_df = pd.read_csv('/content/drive/MyDrive/GMC-Products/'+item_group_excel, sep='\t')
# 测试与处理item group id的空值问题
nan_count = gmc_df['item group id'].isna().sum()
if nan_count > 0:
  print("👉 结论：正是这几个空值，迫使 Pandas 开启了底层保护机制，将整列的整数强行转化为了带小数点的浮点数 (float64)。")
  gmc_df['item group id'] = (
    pd.to_numeric(gmc_df['item group id'], errors='coerce')
    .astype('Int64')
    .astype(str)
    .replace('<NA>', '')
  )

gmc_df['id'] = gmc_df['id'].astype(str)
gmc_df['item group id'] = gmc_df['item group id'].astype(str)

print('GMC item group id data')
print(gmc_df.head()) # 打印出来看看

# 4. 读取Shopify Sales数据
shop_df = pd.read_csv('/content/drive/MyDrive/Shopify-Sales/'+shop_excel)
shop_df['Product ID'] = shop_df['Product ID'].astype(str)
shop_df['Product ID'] = (
    pd.to_numeric(shop_df['Product ID'], errors='coerce')
    .astype('Int64')
    .astype(str)
    .replace('<NA>', '')
)
print('Shopify Sales data')
print(shop_df.head())


# 5. 数据缝合 (VLOOKUP 的 Python 替代版)
# 把 GMC 里的父体 ID 拼接到 Ads 的表现数据上
merged_df = pd.merge(ads_df, gmc_df[['id', 'item group id']],
                     left_on='Item ID', right_on='id', how='left')

merged_df['Clicks'] = pd.to_numeric(merged_df['Clicks'].astype(str).str.replace(r'[^\d.]', '', regex=True), errors='coerce')
merged_df['Conversions'] = pd.to_numeric(merged_df['Conversions'].astype(str).str.replace(r'[^\d.]', '', regex=True), errors='coerce')
merged_df['Impr.'] = pd.to_numeric(merged_df['Impr.'].astype(str).str.replace(r'[^\d.]', '', regex=True), errors='coerce')
merged_df['Cost'] = pd.to_numeric(merged_df['Cost'].astype(str).str.replace(r'[^\d.]', '', regex=True), errors='coerce')
merged_df['Conv. value'] = pd.to_numeric(merged_df['Conv. value'].astype(str).str.replace(r'[^\d.]', '', regex=True), errors='coerce')
# print('Merged data')
print(merged_df.head())


# 6. 按手机壳父体 (Item Group ID) 聚合总花费和总产出
merged_df['item group id'] = merged_df['item group id'].replace('', np.nan)
merged_df['item group id'] = merged_df['item group id'].fillna('Unmatched')
parent_df = merged_df.groupby('item group id').agg(
    total_cost=('Cost', 'sum'),
    total_value=('Conv. value', 'sum'),
    total_clicks=('Clicks', 'sum'),
    total_conv=('Conversions', 'sum'),
    total_impr=('Impr.', 'sum')
).reset_index()
# print(parent_df.head())

import numpy as np
assert np.isclose(merged_df['Cost'].sum(), parent_df['total_cost'].sum(), rtol=1e-10), 'Aggregation后总消耗要相同'

# assert merged_df['Cost'].sum() == parent_df['total_cost'].sum(), 'Aggregation后总消耗要相同'

# 7. 计算Aggerate之后的真实 ROAS
# 避免除以 0 的报错
parent_df['roas'] = parent_df['total_value'] / parent_df['total_cost'].replace(0, 1)


# 8. 定义并应用 HSZV 标签逻辑
minConv = 4
tROAS = 1.2
minClicks=100
minImpr=1000
minCost=25
# hero toas
ht = 1.16
# villains toas
vt = 0.6

# 8.1
# 针对Shopify Data补充parent_df
parent_df = pd.merge(parent_df, shop_df[['Product ID', 'Quantity ordered', 'Gross sales']],
                     left_on='item group id', right_on='Product ID', how='left')
parent_df['shop_roi'] = parent_df['Gross sales']/parent_df['total_cost'].replace(0, 1)


def assign_hszv(row):
    cost = row['total_cost']
    roas = row['roas']
    conv = row['total_conv']
    impr = row['total_impr']
    clicks = row['total_clicks']
    shop_roi = row['shop_roi']
    shop_conv = row['Quantity ordered']
    shop_sales = row['Gross sales']

    # ROAS 超过 优选ROAS
    if roas >= ht * tROAS:
        return 'Heroes' if conv > minConv else ('Sidekicks' if conv > 1 else 'Learning')

    # ROAS 落在 劣等和优选ROAS之间
    if roas >= vt * tROAS:
      if roas >= tROAS and conv >= minConv:
        return 'Sidekicks'

      # if roas < tROAS and conv >= 2*minConv:
      #  return 'Villains'

      return 'Learning'

    # ROAS 落在 劣等ROAS之下
    if conv > minConv:
      return 'Villains'

    # 以下是灰色区域，需要仔细、反复对比琢磨

    # 有了不错的clicks（实际花了钱），但是roas特别低
    if roas <=0.5 * tROAS and cost >= minCost:
      return 'Villains'

    return 'Low'

parent_df['custom_label_0'] = parent_df.apply(assign_hszv, axis=1)

mask_unmatched = parent_df['item group id'].astype(str).str.strip().str.lower() == 'unmatched'


# 1. 条件 A：精准识别真实的 NaN (缺失值)
mask_nan = parent_df['item group id'].isna()

# 2. 条件 B：精准识别文本形式的 'unmatched' (兼容大小写和首尾空格)
mask_text = parent_df['item group id'].astype(str).str.strip().str.lower() == 'unmatched'

# 3. 条件 C（可选扩展）：识别完全空白的字符串 '' (前期清洗可能残留)
mask_empty = parent_df['item group id'].astype(str).str.strip() == 'nan'

# 4. 合并所有掩码（使用位运算符 | 表示“或”）
final_mask = mask_nan | mask_text | mask_empty
parent_df.loc[final_mask, 'custom_label_0'] = 'Unmatched'


def assign_outliers(row):
    values = row['total_value']
    roas = row['roas']
    conv = row['total_conv']

    shop_roi = row['shop_roi']
    shop_conv = row['Quantity ordered']
    shop_sales = row['Gross sales']

    if shop_roi < vt*tROAS or shop_conv < conv or shop_roi < roas or shop_sales < values:
      return 'Warning'

parent_df['comment'] = parent_df.apply(assign_outliers, axis=1)


Google Ads item performancedata
          Item ID                                      Product Title  \
0  47326359847127                                                 --   
1  48219895660759  Magnetic Wireless Charging | Premium Magnetic ...   
2  48219971387607  Magnetic Wireless Charging Phone Case | Premiu...   
3  48161159512279  Slim Ring Magnetic Wireless Charging Phone Cas...   
4  45685230764247  SPICEUP STUDIO | Reflective Phone Case | VS Of...   

  Conv. value / cost Conv. value Currency code Cost Conversions Clicks Impr.  \
0                  0           0           USD    0           0      0     1   
1                  0           0           USD    0           0      0    21   
2                  0           0           USD    0           0      0    22   
3                  0           0           USD    0           0      0    31   
4                  0           0           USD    0           0      0    13   

  Cost / conv.  
0            0  
1            0  
2  

In [28]:
from google.colab import auth
import gspread
from google.auth import default
from datetime import datetime, timezone, timedelta

beijing_tz = timezone(timedelta(hours=8))
time_str = datetime.now(beijing_tz).strftime("%Y%m%d%H%M")
# 在云盘中创建一张全新的 Google Sheet
# sh = gc.create('HSZV_Final_Feed')
sh = gc.create('ItemPerformance-Merged-'+time_str)
worksheet = sh.sheet1

# 把刚才处理好的 final_feed 数据转化为列表格式并写入表格
# [final_feed.columns.values.tolist()] 是写入表头
# final_feed.values.tolist() 是写入所有行数据
worksheet.update([merged_df.columns.values.tolist()] + merged_df.fillna('').values.tolist())
print(f"Merged数据已成功生成: https://docs.google.com/spreadsheets/d/{sh.id}")

sh = gc.create('GroupPerformance-Stats-'+time_str)
worksheet = sh.sheet1
worksheet.update([parent_df.columns.values.tolist()] + parent_df.fillna('').values.tolist())
print(f"Aggregation数据已成功生成: https://docs.google.com/spreadsheets/d/{sh.id}")

Merged数据已成功生成: https://docs.google.com/spreadsheets/d/1GxrNbimZCHoyheHy3PfzgqPoEX5d7eHomLnKHZ7Vakg
Aggregation数据已成功生成: https://docs.google.com/spreadsheets/d/1JID5kWA2W4AkI5GHC_-a0GdviyGoPzi4lH7zJASHURw


In [26]:
from google.colab import auth
import gspread
from google.auth import default

USE_REVIEWED_SHEETS = True

# 1. 确定使用哪个aggregation数据，自动的，还是手动调整过的
if USE_REVIEWED_SHEETS:
  print('Worksheet ID:' + sh.id)
  worksheet = gc.open_by_key(sh.id).sheet1
  rows = worksheet.get_all_values()
  parent_df = pd.DataFrame.from_records(rows[1:], columns=rows[0])

# 9. 整理出最后使用的
final_feed = pd.merge(merged_df[['Item ID', 'item group id']],
                      parent_df[['item group id', 'custom_label_0']],
                      on='item group id', how='left')

# 只保留 GMC 补充馈单需要的两列，并去重
final_feed = final_feed[['Item ID', 'item group id', 'custom_label_0']].drop_duplicates()

# 打印前 5 行让你核对一下结果是否精准
print(final_feed.head())

# 在云盘中创建一张全新的 Google Sheet
# sh = gc.create('HSZV_Final_Feed')
sh = gc.create('HSZV_Feed_stats-'+time_str)
worksheet = sh.sheet1

worksheet.update([final_feed.columns.values.tolist()] + final_feed.fillna('').values.tolist())

print(f"数据已成功生成！表格链接: https://docs.google.com/spreadsheets/d/{sh.id}")

Worksheet ID:1pHhcNxmm2TrIpe5q-dNUJvr-UsE0v1G7yhweYgJSpjg
          Item ID  item group id custom_label_0
0  47326359847127  9076887453911         Heroes
1  48219895660759  9163757158615      Sidekicks
2  48219971387607  9163818467543            Low
3  48161159512279  9148265267415            Low
4  45685230764247  8136663793879            Low
数据已成功生成！表格链接: https://docs.google.com/spreadsheets/d/1Yjs_FMGX6x3DoJi74oHPQpZaqTldA6C_wsuVOpUe24Q
